# 15. Sparse Extended Information Filter (SEIF)

EKF-SLAM은 covariance matrix $\Sigma$를 유지하지만, SEIF는 information form을 사용한다.

$$\Omega=\Sigma^{-1}, \qquad \xi=\Omega\mu$$

핵심 직관은 **멀리 떨어진 landmark 사이의 약한 상관을 제거해서 sparse graph 구조를 유지**하는 것이다. 완전한 EKF-SLAM보다 근사적이지만, 큰 지도에서 계산량을 줄이는 방향을 보여준다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False


## 1. Covariance와 Information Matrix

공분산에서 큰 값은 함께 흔들리는 불확실성을 뜻하고, information matrix의 nonzero 패턴은 직접적인 제약 관계를 뜻한다.

In [ ]:
np.random.seed(15)
n = 9
positions = np.linspace(0, 8, n)
D = np.abs(positions[:, None] - positions[None, :])
Sigma = np.exp(-D / 2.2) + 0.08 * np.eye(n)
Omega = np.linalg.inv(Sigma)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(Sigma, cmap='viridis')
axes[0].set_title('Dense covariance $\\Sigma$')
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(Omega, cmap='coolwarm')
axes[1].set_title('Information matrix $\\Omega=\\Sigma^{-1}$')
fig.colorbar(im1, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xlabel('state index')
    ax.set_ylabel('state index')
plt.tight_layout()
plt.savefig('assets/15_information_form.png', dpi=160)
plt.show()

## 2. Sparsification

SEIF는 모든 상관을 유지하지 않고 작은 off-diagonal 항을 제거한다. 이는 그래프에서 약한 edge를 끊는 것과 같다.

In [ ]:
threshold = 0.08
Omega_sparse = Omega.copy()
mask_offdiag = ~np.eye(n, dtype=bool)
Omega_sparse[mask_offdiag & (np.abs(Omega_sparse) < threshold)] = 0.0
Sigma_approx = np.linalg.inv(Omega_sparse)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].spy(np.abs(Omega) > 1e-10, markersize=8)
axes[0].set_title('Before sparsification')
axes[1].spy(np.abs(Omega_sparse) > 1e-10, markersize=8)
axes[1].set_title('After sparsification')
err = Sigma_approx - Sigma
im = axes[2].imshow(err, cmap='coolwarm')
axes[2].set_title('Approximation error')
fig.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.set_xlabel('state index')
    ax.set_ylabel('state index')
plt.tight_layout()
plt.savefig('assets/15_seif_sparsification.png', dpi=160)
plt.show()

print('original nonzeros:', np.count_nonzero(np.abs(Omega) > 1e-10))
print('sparse nonzeros:', np.count_nonzero(np.abs(Omega_sparse) > 1e-10))
print('relative covariance error:', np.linalg.norm(err) / np.linalg.norm(Sigma))

## 3. 로보틱스 연결

| 개념 | 의미 | 로보틱스 활용 |
|---|---|---|
| Information form | $\Omega,\xi$로 Gaussian 표현 | 큰 SLAM 문제의 sparse 구조 활용 |
| Sparsification | 약한 상관 제거 | 계산량 감소, 근사 오차 발생 |
| Graph view | nonzero가 제약 edge | factor graph / pose graph 최적화와 연결 |

SEIF는 현대 SLAM의 주류 구현이라기보다, **sparsity를 SLAM 계산의 핵심 자원으로 보는 관점**을 익히는 데 중요하다.